# Bus Booking Analytics - ETL Pipeline

This notebook implements the complete **ETL (Extract, Transform, Load)** pipeline as described in the project documentation.
It covers:
1. **Extract**: Loading raw dirty CSV datasets (`bus_booking_raw.csv`, `customers.csv`, `buses.csv`, `routes.csv`) from the local directories.
2. **Transform & Validate**:
   - Flexible Date Parsing using `format='mixed'` to resolve inconsistent date formats.
   - Deduplication (based on `Booking_ID`)
   - Handling Null Values (e.g. dropping bookings with null fares, filling missing customer phone numbers with 'Unknown')
   - Data Standardization (dates formatted to YYYY-MM-DD, proper title casing for names and routes)
   - Data Consistency Checks (Travel Date >= Booking Date, Fare > 0, Seat Number within physical Bus Capacity)
   - Referential Integrity checks between all foreign keys.
3. **Load**: Loading validated data directly into **MySQL** database (`bus_booking_analytics`).

### Step 1: Extract (Load Raw CSVs)

In [25]:
import pandas as pd
import os
import sys

# Import etl_pipeline module from current directory
import etl_pipeline

data_dir = '../data'
bookings, customers, buses, routes = etl_pipeline.load_csv_data(data_dir)
print(f"Raw Bookings count: {len(bookings)}")
print(f"Raw Customers count: {len(customers)}")

Loading CSV files...
Loading raw bookings from ../data\bus_booking_raw.csv
Raw Bookings count: 20592
Raw Customers count: 301


### Step 2: Transform, Clean & Validate
We run the validation logic that details formatting repairs, drop counts, and deduplications.

In [26]:
bookings_clean, customers_clean, buses_clean, routes_clean = etl_pipeline.transform_and_validate(
    bookings, customers, buses, routes
)

--- ETL CLEANING & TRANSFORMATION REPORT ---
Deduplication: Removed 980 duplicate booking records.
Null Handling: Removed 383 booking records with missing Fare_Amount.
Null Handling: Filled 21 missing customer phone numbers with 'Unknown'.
Null Handling/Standardization: Cleansed Gender field, filling 9 missing values.
Null Handling/Outliers: Cleansed Age field, filling 16 nulls and correcting 30 outliers to median (49).
Constraint Check: Removed 373 bookings where Travel_Date was before Booking_Date.
Constraint Check: Removed 288 bookings with non-positive Fare_Amount (e.g. negative values).
Capacity Check: Removed 186 bookings where Seat_Number exceeded Bus Capacity.

--- SUMMARY OF PROCESS ---
Bookings: 20592 raw rows -> 18382 clean rows (Removed 2210 rows).
Customers: 301 raw rows -> 301 clean rows.
--------------------------------------------


### Step 3: Load into MySQL Database
Enter your MySQL connection credentials below to load the clean datasets directly into the database.

In [27]:
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': 'root',  
    'database': 'bus_booking_analytics'
}

etl_pipeline.load_to_mysql(bookings_clean, customers_clean, buses_clean, routes_clean, mysql_config)

Connecting to MySQL...
Successfully loaded datasets into MySQL database.


### Step 4: Verification Queries via SQL
Let's run queries on our cleaned dataset in the MySQL database to generate Key Business Metrics.

In [28]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import mysql.connector

conn = mysql.connector.connect(
    host=mysql_config['host'],
    user=mysql_config['user'],
    password=mysql_config['password'],
    database=mysql_config['database']
)

print("--- 1. KEY METRICS SUMMARY ---")
metrics_query = """
SELECT 
    COUNT(Booking_ID) as Total_Bookings,
    ROUND(SUM(Fare_Amount), 2) as Total_Revenue,
    ROUND(AVG(Fare_Amount), 2) as Avg_Fare
FROM Bookings
"""
print(pd.read_sql_query(metrics_query, conn))

print("\n--- 2. TOP 5 ROUTES BY REVENUE ---")
routes_query = """
SELECT 
    r.Source, 
    r.Destination, 
    COUNT(b.Booking_ID) as Booking_Count,
    SUM(b.Fare_Amount) as Route_Revenue
FROM Bookings b
JOIN Routes r ON b.Route_ID = r.Route_ID
GROUP BY r.Route_ID
ORDER BY Route_Revenue DESC
LIMIT 5;
"""
print(pd.read_sql_query(routes_query, conn))

print("\n--- 3. REVENUE BY BUS TYPE ---")
bus_query = """
SELECT 
    bus.Bus_Type, 
    COUNT(b.Booking_ID) as Booking_Count,
    SUM(b.Fare_Amount) as Revenue
FROM Bookings b
JOIN Buses bus ON b.Bus_ID = bus.Bus_ID
GROUP BY bus.Bus_Type
ORDER BY Revenue DESC;
"""
print(pd.read_sql_query(bus_query, conn))

print("\n--- 4. OCCUPANCY RATE ESTIMATE PER BUS ---")
occupancy_query = """
SELECT 
    b.Bus_ID,
    bus.Bus_Number,
    bus.Capacity,
    COUNT(b.Booking_ID) as Bookings_Made,
    ROUND((COUNT(b.Booking_ID) * 100.0) / (bus.Capacity * 20.0), 2) as Estimated_Occupancy_Percentage
FROM Bookings b
JOIN Buses bus ON b.Bus_ID = bus.Bus_ID
GROUP BY b.Bus_ID
ORDER BY Estimated_Occupancy_Percentage DESC;
"""
print(pd.read_sql_query(occupancy_query, conn))

conn.close()

--- 1. KEY METRICS SUMMARY ---
   Total_Bookings  Total_Revenue  Avg_Fare
0           18382    15532689.13    844.99

--- 2. TOP 5 ROUTES BY REVENUE ---
      Source Destination  Booking_Count  Route_Revenue
0    Chennai   Hyderabad           2013     2403683.58
1  Hyderabad   Bangalore           2012     2245826.97
2     Mumbai   Ahmedabad           1969     1971510.92
3  Bangalore   Hyderabad           1778     1931700.02
4       Pune         Goa           1619     1490080.37

--- 3. REVENUE BY BUS TYPE ---
         Bus_Type  Booking_Count     Revenue
0  Non-AC Sleeper           4904  4631586.42
1       AC Seater           5367  4109834.53
2      AC Sleeper           4328  3879989.10
3   Non-AC Seater           3783  2911279.08

--- 4. OCCUPANCY RATE ESTIMATE PER BUS ---
    Bus_ID    Bus_Number  Capacity  Bookings_Made  \
0      209  MH-34-H-5781        40           2129   
1      206  TS-70-C-2890        40           1683   
2      201  UP-48-J-4135        50           2097   
3   